# Simulation tutorial

This notebook works through pyQuil's JAX-based simulators in five steps:

1. **A simple program.** Compute the state vector of a gate-only circuit.
2. **A variational algorithm.** Optimize a parametric circuit with exact gradients from `jax.grad`.
3. **A noisy program.** Compute the exact outcome distribution with the density-matrix simulator, then draw shots
   with the trajectory simulator and watch their statistics converge to the exact values.
4. **The variational algorithm under noise.** Find out which kinds of noise move the optimum.
5. **Performance.** See where `jax.jit` pays off.

See [Simulating programs](simulation.rst) for a reference to the simulator API, and [Noise models](noise.rst) for
how to build channels and noise models.

> **Qubit ordering.** The simulators are **big-endian**: the first entry of the simulator's `qubits` list is the
> *most* significant subsystem. With `qubits=[0, 1]`, the program `X 0` gives the basis state `|10>`.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)

import math

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import quax as qx

from pyquil.gates import CNOT, CZ, H, MEASURE, RX, RY, RZ
from pyquil.quilatom import MemoryReference
from pyquil.quilbase import Declare
from pyquil.quil import Program
from pyquil.noise import Channel, MeasurementChannel, NoiseModel, SuperopChannel
from pyquil.simulation import (
    DensityMatrixSimulator,
    PureStateVectorSimulator,
    TrajectorySimulator,
)

## 1. Simple Program — State Vector

We create a 2-qubit Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$ using a Hadamard and a CNOT.

In [ ]:
bell_program = Program(H(0), CNOT(0, 1))
print(bell_program)

`PureStateVectorSimulator` takes a program made only of gates, with no measurements or resets, and returns a
`qx.StateVector`. This program has no runtime parameters, so `compute` is called with no arguments.

In [ ]:
sim = PureStateVectorSimulator(bell_program)
psi = sim.compute()

print(f"State vector: {psi.matrix}")
print(f"Qubit dims:   {psi.dims}")
print(f"Probabilities: {jnp.abs(psi.matrix) ** 2}")

We can verify this is the correct Bell state by computing the expectation value
of the $Z \otimes Z$ observable.  For $|\Phi^+\rangle$, we expect $\langle ZZ \rangle = 1$.

In [ ]:
Z_mat = jnp.array([[1, 0], [0, -1]], dtype=complex)
ZZ = qx.Observable.from_matrix(jnp.kron(Z_mat, Z_mat), dims=((2, 2), (2, 2)))

print(f"<ZZ> = {qx.estimate(psi, ZZ):.6f}")

The simulator is also compatible with `jax.jit` for fast repeated evaluations:

In [ ]:
compute_jit = jax.jit(sim.compute)

psi_jit = compute_jit()
print(f"JIT result matches: {float(qx.fidelity(psi, psi_jit)):.10f}")

We can also extract the full program unitary:

In [ ]:
U = sim.unitary()
print("Program unitary:")
print(np.array(U.matrix).round(4))

## 2. Variational Optimization — VQE with the State-Vector Simulator

A variational eigensolver minimizes $E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle$ over
circuit parameters. Because `compute` is differentiable, the gradient comes from `jax.grad`
rather than from parameter-shift rules or finite differences, and the whole optimization step
can be `jit`-compiled.

Our target is the two-qubit transverse-field Ising model

$$H = Z_0 Z_1 + h\,(X_0 + X_1),$$

whose ground-state energy we can get exactly by diagonalizing a $4\times4$ matrix — a ground
truth to check the optimizer against.

In [ ]:
h_field = 0.6

# `Z_mat` came from section 1; define the other two Paulis alongside it so later sections
# (which build ZI / IZ observables) can reuse them.
I_mat = jnp.eye(2, dtype=complex)
X_mat = jnp.array([[0, 1], [1, 0]], dtype=complex)

Z_np, X_np, I_np = np.asarray(Z_mat), np.asarray(X_mat), np.asarray(I_mat)
H_np = np.kron(Z_np, Z_np) + h_field * (np.kron(X_np, I_np) + np.kron(I_np, X_np))
H_tfim = qx.Observable.from_matrix(jnp.asarray(H_np), dims=((2, 2), (2, 2)))

exact_energies = np.linalg.eigvalsh(H_np)
exact_ground = float(exact_energies[0])

print(f"H spectrum:            {exact_energies.round(6)}")
print(f"Exact ground energy:   {exact_ground:.8f}")
print(f"Tr(H) = {np.trace(H_np).real:.1f}   (needed later — it makes the noise analysis clean)")


### A native-gate ansatz

We write the ansatz the way hardware actually runs it. On a Rigetti QPU the only continuously
parameterized gate is `RZ`, which is *virtual* — a frame change, so it costs no time and carries
no noise. Every physical rotation is a fixed `RX(±pi/2)` pulse. The identity

$$R_Y(\theta) = R_X(\pi/2)\, R_Z(\theta)\, R_X(-\pi/2)$$

lets us build an $R_Y$ rotation with all of the parameter dependence inside the virtual `RZ`.

That structure matters for section 4: a noise channel is keyed to a concrete instruction, so it
can attach to the fixed `RX(±pi/2)` and `CZ` gates but *not* to a runtime-parametric gate. Writing
the ansatz natively is what makes the noisy re-optimization below possible at all.

In [ ]:
HALF_PI = np.pi / 2
VQE_QUBITS = (0, 1)
N_LAYERS = 2


def native_ry(program: Program, angle, qubit: int) -> Program:
    """Append RY(angle) as RX(pi/2) . RZ(angle) . RX(-pi/2) — parameter in the virtual RZ."""
    program += RX(HALF_PI, qubit)
    program += RZ(angle, qubit)
    program += RX(-HALF_PI, qubit)
    return program


def vqe_ansatz() -> Program:
    """Layers of RY rotations (one shared angle per layer) separated by a CZ entangler."""
    program = Program(Declare("theta", "REAL", N_LAYERS))
    for layer in range(N_LAYERS):
        angle = MemoryReference("theta", layer)
        for qubit in VQE_QUBITS:
            program = native_ry(program, angle, qubit)
        if layer < N_LAYERS - 1:
            program += CZ(*VQE_QUBITS)
    return program


print(vqe_ansatz())


### The flat parameter vector

`compute` takes a flat vector with **one slot for each distinct memory reference** the program's gates read, in
the order each is first used. Each of the two layer angles is shared by two qubits, but each still occupies a single
slot, so the vector is just `[theta[0], theta[1]]`.

`sim.parameters` lists the layout, and `sim.linearize` builds the vector from a memory map. The gradient with
respect to a slot is already the total derivative through every gate that reads it.

In [ ]:
vqe_sim = PureStateVectorSimulator(vqe_ansatz(), qubits=list(VQE_QUBITS))

print(f"Parameter layout: {vqe_sim.parameters}")
print(f"Vector length:    {len(vqe_sim.parameters)}")
print(f"linearize({{'theta': [0.3, 0.7]}}) = {np.asarray(vqe_sim.linearize({'theta': [0.3, 0.7]}))}")

### Energy, gradient, and a gradient check

In [ ]:
def energy_ideal(angles):
    """<psi(angles)|H|psi(angles)> with no noise."""
    return jnp.real(qx.estimate(vqe_sim.compute(vqe_sim.linearize({"theta": angles})), H_tfim))


angles0 = jnp.array([0.4, 0.9])
grad_analytic = jax.grad(energy_ideal)(angles0)

step = 1e-6
grad_fd = np.array([
    float(energy_ideal(angles0.at[i].add(step)) - energy_ideal(angles0.at[i].add(-step))) / (2 * step)
    for i in range(N_LAYERS)
])

print(f"E(angles0)        = {float(energy_ideal(angles0)):+.8f}")
print(f"jax.grad          = {np.asarray(grad_analytic).round(8)}")
print(f"finite difference = {grad_fd.round(8)}")
print(f"max difference    = {np.max(np.abs(np.asarray(grad_analytic) - grad_fd)):.2e}")

### Optimize

Plain gradient descent on a `jit`-compiled value-and-gradient. The ansatz is small enough to have
local minima, so we restart from several random points and keep the best.

In [ ]:
def optimize(cost, init, steps: int = 800, learning_rate: float = 0.05):
    """Gradient descent on `cost`; returns (optimal angles, energy history)."""
    value_and_grad = jax.jit(jax.value_and_grad(cost))
    angles = jnp.asarray(init, dtype=float)
    history = []
    for _ in range(steps):
        value, gradient = value_and_grad(angles)
        history.append(float(value))
        angles = angles - learning_rate * gradient
    return np.asarray(angles), history


def optimize_multistart(cost, n_starts: int = 8, seed: int = 0):
    """Restart from several random points and keep the lowest minimum found."""
    best = None
    for start in range(n_starts):
        init = np.random.default_rng(seed + start).uniform(-np.pi, np.pi, N_LAYERS)
        angles, history = optimize(cost, init)
        if best is None or history[-1] < best[1][-1]:
            best = (angles, history)
    return best


ideal_angles, ideal_history = optimize_multistart(energy_ideal)
ideal_energy = ideal_history[-1]

print(f"Optimal angles:  {ideal_angles.round(6)}")
print(f"VQE energy:      {ideal_energy:.8f}")
print(f"Exact ground:    {exact_ground:.8f}")
print(f"Error:           {abs(ideal_energy - exact_ground):.2e}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(ideal_history, linewidth=1.5, label="VQE energy")
ax.axhline(exact_ground, color="red", linestyle="--", linewidth=1.2, label="exact ground state")
ax.set_xlabel("Gradient-descent step")
ax.set_ylabel(r"$\langle H \rangle$")
ax.set_title("VQE convergence (state-vector simulator)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


### The cost landscape

With only two parameters we can plot $E(\theta_0, \theta_1)$ in full. `jax.vmap` evaluates the
whole grid in one batched call. We reuse this landscape in section 4 to see what noise does to
it.

In [ ]:
grid_1d = jnp.linspace(-np.pi, np.pi, 81)
grid_a, grid_b = jnp.meshgrid(grid_1d, grid_1d, indexing="ij")
grid_points = jnp.stack([grid_a.ravel(), grid_b.ravel()], axis=-1)


def landscape(cost):
    """Evaluate `cost` on the parameter grid, returning a 2-D array."""
    return np.asarray(jax.vmap(cost)(grid_points)).reshape(grid_a.shape)


landscape_ideal = landscape(energy_ideal)

fig, ax = plt.subplots(figsize=(5.6, 4.6))
contour = ax.contourf(np.asarray(grid_a), np.asarray(grid_b), landscape_ideal, levels=30, cmap="viridis")
ax.plot(*ideal_angles, "r*", markersize=16, label=f"optimum {ideal_angles.round(3)}")
ax.set_xlabel(r"$\theta_0$")
ax.set_ylabel(r"$\theta_1$")
ax.set_title(r"Ideal cost landscape $\langle H\rangle(\theta_0,\theta_1)$")
ax.legend(loc="upper right")
fig.colorbar(contour, ax=ax, shrink=0.85)
fig.tight_layout()
plt.show()


## 3. A Noisy Program

Real hardware has imperfect gates and measurements. In this section we build a noise model that assigns a
fidelity to each native instruction, as a device's instruction set architecture (ISA) reports it, and simulate a
Bell-state program two ways:

| Simulator | Method | Result | Memory |
|-----------|--------|--------|--------|
| `DensityMatrixSimulator` | Superoperator evolution | Exact outcome distribution | $O(4^n)$ |
| `TrajectorySimulator` | Monte-Carlo trajectories | Sampled shots | $O(2^n)$ per trajectory |

### Build a noise model

The device has native `RX(±pi/2)`, `RX(pi)` and `CZ` gates and `MEASURE`, each with a realistic fidelity. Every
channel is keyed to one concrete instruction.

* `Channel.from_gate_fidelity` turns an average gate fidelity into a depolarizing channel.
* `MeasurementChannel.from_readout_fidelity` models readout (classification) error.

For a real QPU, `NoiseModel.from_isa` builds the same kind of model directly from the device's QCS
`InstructionSetArchitecture`.

In [ ]:
GATE_FIDELITIES = {0: 0.995, 1: 0.993}     # single-qubit RX fidelity per qubit
READOUT_FIDELITIES = {0: 0.97, 1: 0.95}
CZ_FIDELITY = 0.96

channels = []
for qubit, fidelity in GATE_FIDELITIES.items():
    for angle in (math.pi, math.pi / 2, -math.pi / 2):
        channels.append(Channel.from_gate_fidelity(RX(angle, qubit), fidelity=fidelity))
    channels.append(MeasurementChannel.from_readout_fidelity(MEASURE(qubit, None), fidelity=READOUT_FIDELITIES[qubit]))
channels.append(Channel.from_gate_fidelity(CZ(0, 1), fidelity=CZ_FIDELITY))

noise_model = NoiseModel.from_channels(channels)

print(f"{len(noise_model.channels)} noise channels:")
for instruction in noise_model.channels:
    print(f"  {instruction}")

### Build a Bell-state program in native gates

The device supports only `RX`, `RZ` and `CZ`, so we decompose the textbook Hadamard + CNOT circuit into native
operations:

$$H = R_Z(\pi)\, R_X(\pi/2)\, R_Z(\pi/2)$$
$$\text{CNOT}_{01} = R_X(\pi/2)_1 \cdot CZ_{01} \cdot R_X(-\pi/2)_1$$

In [ ]:
noisy_bell = Program()
noisy_bell += Declare("ro", "BIT", 2)

# Hadamard on qubit 0 (native decomposition)
noisy_bell += RZ(math.pi, 0)
noisy_bell += RX(math.pi / 2, 0)
noisy_bell += RZ(math.pi / 2, 0)

# CNOT(0,1) via CZ + local rotations
noisy_bell += RX(math.pi / 2, 1)
noisy_bell += CZ(0, 1)
noisy_bell += RX(-math.pi / 2, 1)

# Measurement
noisy_bell += MEASURE(0, MemoryReference("ro", 0))
noisy_bell += MEASURE(1, MemoryReference("ro", 1))

print(noisy_bell)

### Density-matrix simulation (exact)

Channels are keyed by *instruction*, so a channel applies only where one of the program's instructions matches its
key exactly. Two consequences are worth noting:

- The noise model has no `RZ` entries, so the program's `RZ(pi)` and `RZ(pi/2)` gates are ideal. That matches the
  hardware rather than leaving a gap: `RZ` on Rigetti hardware is *virtual*, a frame change with no pulse, and so
  it has no duration and no noise.
- For the same reason, a noise channel never applies to a gate whose angle is a runtime parameter. Every other
  rotation is calibrated at a fixed set of angles in practice.

In [ ]:
dm_sim = DensityMatrixSimulator(noisy_bell, noise_model=noise_model)
rho = jax.jit(dm_sim.compute)()

print(f"Density matrix shape: {rho.matrix.shape}")
print(f"Trace: {jnp.trace(rho.matrix).real:.8f}")
print(f"Purity Tr(rho^2): {jnp.trace(rho.matrix @ rho.matrix).real:.6f}")

In [ ]:
# Observables. With qubits=[0, 1] and big-endian ordering, the *left* factor of a
# Kronecker product acts on qubit 0.
ZI = qx.Observable.from_matrix(jnp.kron(Z_mat, I_mat), dims=((2, 2), (2, 2)))
IZ = qx.Observable.from_matrix(jnp.kron(I_mat, Z_mat), dims=((2, 2), (2, 2)))

dm_zz = float(qx.estimate(rho, ZZ))
dm_zi = float(qx.estimate(rho, ZI))
dm_iz = float(qx.estimate(rho, IZ))

print(f"DM  <Z0 Z1> = {dm_zz:+.6f}")
print(f"DM  <Z0>    = {dm_zi:+.6f}")
print(f"DM  <Z1>    = {dm_iz:+.6f}")


### Exact outcome probabilities and finite shots

`compute` applies each `MEASURE` as its outcome-averaged channel, so the state above carries no readout error: the
expectation values just computed belong to the *state*. What an experiment records is different: bits that have
passed through the readout channel.

* `outcome_probabilities` gives the exact distribution of those bits, with readout error included.
* `TrajectorySimulator` samples shots from the same distribution.

Comparing the two separates the two sources of error. The *noise* shifts the exact value away from the ideal one,
and *shot noise* scatters an estimate around the exact value.

In [ ]:
probs = np.asarray(dm_sim.outcome_probabilities())   # probs[a, b] = P(q0 reads a, q1 reads b)
print(f"Measured qubits: {dm_sim.measured_qubits}")

# Exact expectation values of the recorded bits, readout error included.
signs = np.array([1.0, -1.0])
exact_zz = float(np.einsum("ab,a,b->", probs, signs, signs))
exact_zi = float(np.einsum("ab,a->", probs, signs))
exact_iz = float(np.einsum("ab,b->", probs, signs))

traj_sim = TrajectorySimulator(noisy_bell, noise_model=noise_model)
num_shots = 10_000
outcomes = np.asarray(traj_sim.sample(num_trajectories=num_shots, key=jax.random.key(42)))
indices = 2 * outcomes[:, 0] + outcomes[:, 1]   # big-endian basis index

print(f"Outcome distribution (exact):     {probs.ravel().round(4)}")
print(f"Outcome distribution (10k shots): {np.bincount(indices, minlength=4) / num_shots}")
print(f"Outcomes shape: {outcomes.shape}  (shots x measurements)")
print(f"First 8 shots:\n{outcomes[:8]}")

In [ ]:
# Convert measurement outcomes to observable eigenvalues: +1 for |0>, -1 for |1>
z0_eigenvalues = (-1.0) ** outcomes[:, 0]
z1_eigenvalues = (-1.0) ** outcomes[:, 1]

shot_zz = float(np.mean(z0_eigenvalues * z1_eigenvalues))
shot_zi = float(np.mean(z0_eigenvalues))
shot_iz = float(np.mean(z1_eigenvalues))

print(f"Shots <Z0 Z1> = {shot_zz:+.6f}  (exact: {exact_zz:+.6f},  state: {dm_zz:+.6f})")
print(f"Shots <Z0>    = {shot_zi:+.6f}  (exact: {exact_zi:+.6f},  state: {dm_zi:+.6f})")
print(f"Shots <Z1>    = {shot_iz:+.6f}  (exact: {exact_iz:+.6f},  state: {dm_iz:+.6f})")
print(f"\nExpected 1-sigma shot noise at {num_shots} shots: {1 / np.sqrt(num_shots):.4f}")

### Convergence

The running estimate of $\langle Z_0 Z_1\rangle$ converges to the exact value, readout error included, as
$1/\sqrt{N}$.

In [ ]:
zz_eigenvalues = np.array(z0_eigenvalues * z1_eigenvalues)
cumulative_mean = np.cumsum(zz_eigenvalues) / np.arange(1, len(zz_eigenvalues) + 1)

shots = np.arange(1, len(zz_eigenvalues) + 1)
# Standard error of the running mean.
running_var = np.cumsum(zz_eigenvalues**2) / shots - cumulative_mean**2
stderr = np.sqrt(np.clip(running_var, 0.0, None) / shots)

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(
    shots,
    cumulative_mean - 2 * stderr,
    cumulative_mean + 2 * stderr,
    alpha=0.2,
    label=r"$\pm 2\sigma$",
)
ax.plot(shots, cumulative_mean, linewidth=0.8, label="Shot estimate")
ax.axhline(exact_zz, color="red", linestyle="--", linewidth=1.5, label=f"Exact = {exact_zz:.4f}")
ax.set_xlabel("Number of shots")
ax.set_ylabel(r"$\langle Z_0 Z_1 \rangle$")
ax.set_title(r"Shot-noise convergence of $\langle Z_0 Z_1 \rangle$")
ax.legend()
ax.set_xscale("log")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

### Visualize the noisy density matrix

Finally, we plot $|\rho_{ij}|$ to see how noise degrades the ideal Bell state.

In [ ]:
rho_np = np.array(rho.matrix)
labels = ["00", "01", "10", "11"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im0 = axes[0].imshow(np.abs(rho_np), vmin=0, vmax=0.5, cmap="Blues")
axes[0].set_xticks(range(4), labels)
axes[0].set_yticks(range(4), labels)
axes[0].set_title(r"$|\rho_{ij}|$ (noisy)")
fig.colorbar(im0, ax=axes[0], shrink=0.8)

# Ideal Bell state for comparison
ideal_psi = PureStateVectorSimulator(Program(H(0), CNOT(0, 1))).compute()
ideal_rho = np.outer(np.array(ideal_psi.matrix), np.conj(np.array(ideal_psi.matrix)))

im1 = axes[1].imshow(np.abs(ideal_rho), vmin=0, vmax=0.5, cmap="Blues")
axes[1].set_xticks(range(4), labels)
axes[1].set_yticks(range(4), labels)
axes[1].set_title(r"$|\rho_{ij}|$ (ideal)")
fig.colorbar(im1, ax=axes[1], shrink=0.8)

fig.suptitle("Density matrix: noisy vs. ideal", fontsize=13)
fig.tight_layout()
plt.show()

## 4. Variational Optimization Under Noise — Where Does the Optimum Move?

Now re-run the section 2 optimization on the `DensityMatrixSimulator` with noise, and ask a
sharper question than "how much worse is the energy?": **does the optimum itself move?**

That matters practically. If noise only rescales the cost, the angles you calibrate on a
simulator remain the right angles on hardware, and the variational loop is *self-correcting*. If
noise moves the optimum, the loop chases a different minimum than the one you intended.

We compare three noise models, all built from the same channel API:

| model | channels |
|-------|----------|
| **global depolarizing** | one 2-qubit depolarizing channel on `CZ` — it spans the whole register |
| **local depolarizing** | 1-qubit depolarizing on each fixed `RX(±pi/2)` |
| **coherent** | a systematic `RX` over-rotation plus a residual `ZZ` phase on `CZ` |

The distinction between the two depolarizing models turns out to be the crux.

### Building the channels

`Channel.from_depolarizing_constant` gives the incoherent part. For the coherent part we build a
`SuperopChannel` whose `process` is the *miscalibrated* gate while its `ideal_unitary` stays the
gate we meant to apply — that is exactly what a coherent error is. Composing the two with `@`
gives depolarizing on top of an over-rotation.

In [ ]:
DEPOL_GLOBAL = 0.95      # 2-qubit depolarizing constant on CZ
DEPOL_LOCAL = 0.99       # 1-qubit depolarizing constant on each RX(+-pi/2)
RX_MISCAL = 0.05         # fractional RX over-rotation (5% amplitude error)
CZ_ZZ_PHASE = 0.05       # residual ZZ coupling on CZ, in radians

RX_INSTRUCTIONS = [RX(sign * HALF_PI, q) for q in VQE_QUBITS for sign in (1, -1)]
CZ_INSTRUCTION = CZ(*VQE_QUBITS)
CZ_IDEAL = qx.gates.QUANTUM_GATES["CZ"]


def zz_phase_unitary(phase: float) -> qx.Unitary:
    """exp(-i * phase/2 * Z0 Z1): a residual always-on coupling."""
    diagonal = np.exp(-0.5j * phase * np.array([1.0, -1.0, -1.0, 1.0]))
    return qx.Unitary.from_matrix(jnp.asarray(np.diag(diagonal)), ((2, 2), (2, 2)))


def global_depolarizing_model() -> NoiseModel:
    """A single depolarizing channel spanning the whole register."""
    return NoiseModel.from_channels(
        [Channel.from_depolarizing_constant(CZ_INSTRUCTION, DEPOL_GLOBAL)]
    )


def local_depolarizing_model() -> NoiseModel:
    """Independent single-qubit depolarizing on each physical RX pulse."""
    return NoiseModel.from_channels(
        [Channel.from_depolarizing_constant(inst, DEPOL_LOCAL) for inst in RX_INSTRUCTIONS]
    )


def coherent_model() -> NoiseModel:
    """Systematic over-rotation on every RX pulse, plus a residual ZZ phase on CZ."""
    channels = []
    for inst in RX_INSTRUCTIONS:
        angle = inst.params[0].real
        channels.append(
            SuperopChannel(
                inst=inst,
                process=qx.to_superop(qx.gates.RX(angle * (1 + RX_MISCAL))),
                ideal_unitary=qx.gates.RX(angle),
            )
        )
    channels.append(
        SuperopChannel(
            inst=CZ_INSTRUCTION,
            process=qx.to_superop(zz_phase_unitary(CZ_ZZ_PHASE) @ CZ_IDEAL),
            ideal_unitary=CZ_IDEAL,
        )
    )
    return NoiseModel.from_channels(channels)


# A coherent error is unitary: it costs fidelity without costing purity.
example = coherent_model().get_channel(RX_INSTRUCTIONS[0])
print(f"coherent RX channel:  process_fidelity = {example.process_fidelity:.6f}")
print(f"                      unitarity        = {example.unitarity:.6f}  (1.0 => purely coherent)")
print(f"                      stochastic part  = {example.stochastic_infidelity:.2e}")


### Re-optimize under each model

In [ ]:
def noisy_energy(noise_model):
    """Build <H> for the ansatz evaluated on the density-matrix simulator."""
    sim = DensityMatrixSimulator(vqe_ansatz(), qubits=list(VQE_QUBITS), noise_model=noise_model)

    def cost(angles):
        return jnp.real(qx.estimate(sim.compute(sim.linearize({"theta": angles})), H_tfim))

    return cost


models = {
    "ideal": energy_ideal,
    "global depolarizing": noisy_energy(global_depolarizing_model()),
    "local depolarizing": noisy_energy(local_depolarizing_model()),
    "coherent": noisy_energy(coherent_model()),
}

results = {}
for name, cost in models.items():
    # Start every run from the ideal optimum: we want the *displacement* noise induces,
    # not whatever minimum a random restart happens to land in.
    optimum, history = optimize(cost, ideal_angles)
    results[name] = {
        "angles": optimum,
        "energy": history[-1],
        "shift": float(np.linalg.norm(optimum - ideal_angles)),
        "energy_at_ideal_angles": float(cost(jnp.asarray(ideal_angles))),
    }

header = f"{'model':22s} {'theta*':>22s} {'|shift|':>10s} {'E*':>12s} {'E/E_ideal':>10s}"
print(header)
print("-" * len(header))
for name, r in results.items():
    theta_str = np.array2string(r["angles"].round(5), separator=", ")
    print(f"{name:22s} {theta_str:>22s} {r['shift']:10.6f} {r['energy']:+12.6f} "
          f"{r['energy'] / ideal_energy:10.4f}")

### Reading the table

**Global depolarizing does not move the optimum at all** — the shift is zero to machine
precision, and the energy is scaled by exactly the depolarizing constant. That is a theorem, not a
coincidence. A register-wide depolarizing channel is unital and covariant, so it commutes through
every later gate and can be pushed to the end of the circuit:

$$\rho \mapsto p\,\rho + (1-p)\frac{I}{d}
\quad\Longrightarrow\quad
E_{\text{noisy}}(\theta) = p\,E_{\text{ideal}}(\theta) + (1-p)\frac{\mathrm{Tr}\,H}{d}.$$

With $\mathrm{Tr}\,H = 0$ (printed in section 2) that is just $p\,E_{\text{ideal}}(\theta)$ — an
affine rescaling, which cannot move an argmin. The landscape is flattened toward zero, but every
minimum stays exactly where it was.

**Local depolarizing does move it.** A single-qubit channel is only covariant under single-qubit
gates; it does *not* commute past the `CZ`, so it cannot be pushed to the end and the argument
above fails. This is worth flagging because the folk version of the claim — "depolarizing noise
just shrinks the landscape" — is only true for the register-wide case.

**Coherent errors move it too, and much more cheaply.** Compare the shift against the energy
actually lost: that is the useful figure of merit.

In [ ]:
print(f"{'model':22s} {'|shift|':>10s} {'energy lost':>12s} {'shift per unit loss':>21s}")
print("-" * 68)
for name, r in results.items():
    if name == "ideal":
        continue
    lost = 1.0 - r["energy"] / ideal_energy
    ratio = r["shift"] / lost if lost > 1e-12 else 0.0
    print(f"{name:22s} {r['shift']:10.6f} {lost:12.4f} {ratio:21.3f}")

print()
print("Re-optimizing recovers only a little energy — the noise floor dominates:")
for name, r in results.items():
    if name == "ideal":
        continue
    gain = r["energy_at_ideal_angles"] - r["energy"]
    print(f"  {name:22s} E(ideal angles) = {r['energy_at_ideal_angles']:+.6f} "
          f"-> E* = {r['energy']:+.6f}   (gain {gain:.2e})")


### Seeing it in the landscape

Contours of the ideal and coherent-error landscapes, with both optima marked. The coherent
landscape is *deformed*, not merely scaled, so its minimum sits somewhere else.

In [ ]:
landscape_coherent = landscape(models["coherent"])
landscape_global = landscape(models["global depolarizing"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), sharex=True, sharey=True)
panels = [
    ("Ideal", landscape_ideal, results["ideal"]["angles"]),
    (f"Global depolarizing (p={DEPOL_GLOBAL})", landscape_global, results["global depolarizing"]["angles"]),
    ("Coherent errors", landscape_coherent, results["coherent"]["angles"]),
]

for ax, (title, surface, optimum) in zip(axes, panels):
    contour = ax.contourf(np.asarray(grid_a), np.asarray(grid_b), surface, levels=30, cmap="viridis")
    ax.plot(*ideal_angles, "w*", markersize=15, markeredgecolor="black", label="ideal optimum")
    ax.plot(*optimum, "rx", markersize=11, markeredgewidth=2.5, label="this model's optimum")
    ax.set_title(title)
    ax.set_xlabel(r"$\theta_0$")
    fig.colorbar(contour, ax=ax, shrink=0.85)

axes[0].set_ylabel(r"$\theta_1$")
axes[2].legend(loc="upper right", fontsize=8)
fig.suptitle("Noise reshapes the cost landscape; only coherent errors relocate the minimum", y=1.02)
fig.tight_layout()
plt.show()


### How the shift scales

The single points above are one slice through a trend. Sweeping each error's strength separates
the two behaviours cleanly: the global-depolarizing curve sits flat on zero for *any* strength,
while the local-depolarizing and coherent curves grow from the origin.

In [ ]:
strengths = np.linspace(0.0, 0.20, 7)
sweep = {"global depolarizing": [], "local depolarizing": [], "coherent": []}

for strength in strengths:
    p = 1.0 - strength
    sweep["global depolarizing"].append(
        NoiseModel.from_channels([Channel.from_depolarizing_constant(CZ_INSTRUCTION, p)])
        if strength > 0 else None
    )
    sweep["local depolarizing"].append(
        NoiseModel.from_channels(
            [Channel.from_depolarizing_constant(inst, p) for inst in RX_INSTRUCTIONS]
        )
        if strength > 0 else None
    )
    coherent_channels = [
        SuperopChannel(
            inst=inst,
            process=qx.to_superop(qx.gates.RX(inst.params[0].real * (1 + strength))),
            ideal_unitary=qx.gates.RX(inst.params[0].real),
        )
        for inst in RX_INSTRUCTIONS
    ]
    sweep["coherent"].append(NoiseModel.from_channels(coherent_channels) if strength > 0 else None)

shifts = {name: [] for name in sweep}
for name, model_list in sweep.items():
    for noise_model in model_list:
        if noise_model is None:
            shifts[name].append(0.0)
            continue
        optimum, _ = optimize(noisy_energy(noise_model), ideal_angles)
        shifts[name].append(float(np.linalg.norm(optimum - ideal_angles)))
    print(f"{name:22s} {np.array(shifts[name]).round(5)}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
styles = {"global depolarizing": ("o-", "tab:blue"),
          "local depolarizing": ("s-", "tab:orange"),
          "coherent": ("^-", "tab:red")}
for name, (marker, color) in styles.items():
    ax.plot(strengths, shifts[name], marker, color=color, label=name)

ax.set_xlabel("Error strength  (1 - p  for depolarizing;  fractional over-rotation for coherent)")
ax.set_ylabel(r"$\|\theta^*_{\rm noisy} - \theta^*_{\rm ideal}\|$")
ax.set_title("Displacement of the variational optimum")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


### Takeaway

- A **register-wide depolarizing** channel rescales the cost function and leaves every optimum
  exactly in place. A variational loop is genuinely immune to it — it only loses signal.
- **Local depolarizing** breaks that immunity, because a single-qubit channel does not commute
  past an entangling gate. The optimum does move.
- **Coherent errors** move the optimum the most per unit of energy lost: they deform the
  landscape rather than flattening it. A miscalibration too small to notice in a fidelity number
  can still steer a variational optimizer to the wrong angles — and, because the optimizer
  *absorbs* part of the error, the final energy looks better than the calibration deserves.

The practical reading: track coherent error separately from total infidelity. The channel API
exposes exactly that split via `unitarity`, `coherent_infidelity` and `stochastic_infidelity`.

In [ ]:
print(f"{'gate':34s} {'process_fid':>12s} {'coherent':>10s} {'stochastic':>12s} {'unitarity':>10s}")
print("-" * 82)
for name, noise_model in [("coherent", coherent_model()),
                          ("local depolarizing", local_depolarizing_model())]:
    channel = noise_model.get_channel(RX_INSTRUCTIONS[0])
    print(f"{name + ' / RX(pi/2)':34s} {channel.process_fidelity:12.6f} "
          f"{channel.coherent_infidelity:10.2e} {channel.stochastic_infidelity:12.2e} "
          f"{channel.unitarity:10.6f}")


## 5. Performance scaling

Two costs matter for these simulators, and they dominate in opposite regimes:

- **Python and dispatch overhead**, paid once per operation per call. Constant in the state size,
  so it dominates for small registers.
- **The linear algebra itself**, which grows as $2^n$ (state vector) or $4^n$ (density matrix).

`jax.jit` removes almost all of the first and none of the second, so the speedup it buys depends
strongly on the register size. The sweep below makes that concrete.


### A brickwork ansatz to scale up

In [ ]:
def build_program(num_qubits: int, num_layers: int) -> Program:
    """A brickwork ansatz: RZ-RX(pi/2)-RZ-RX(pi/2)-RZ on every qubit, then two CZ layers."""
    qubits = list(range(num_qubits))
    edges_a = [(i, (i + 1) % num_qubits) for i in range(0, num_qubits, 2)]
    edges_b = [(i, (i + 1) % num_qubits) for i in range(1, num_qubits, 2)]

    program = Program()
    program.declare("theta", "REAL", num_qubits * num_layers * 3)
    offset = 0
    for _ in range(num_layers):
        for q in qubits:
            program += RZ(MemoryReference("theta", offset), q)
            program += RX(0.5 * np.pi, q)
            program += RZ(MemoryReference("theta", offset + 1), q)
            program += RX(0.5 * np.pi, q)
            program += RZ(MemoryReference("theta", offset + 2), q)
            offset += 3
        for i, j in edges_a:
            program += CZ(i, j)
        for i, j in edges_b:
            program += CZ(i, j)
    return program


### Where `jax.jit` helps

Each configuration is timed eagerly and under `jax.jit`, taking the best of three runs after a
warm-up call (so JIT *compile* time is excluded — it is paid once, then amortised over every
subsequent call).


In [ ]:
import time


def best_of(fn, params, reps: int = 3) -> float:
    """Best wall-clock time of `reps` runs, after a warm-up call."""
    fn(params).matrix.block_until_ready()
    times = []
    for _ in range(reps):
        start = time.perf_counter()
        fn(params).matrix.block_until_ready()
        times.append(time.perf_counter() - start)
    return min(times)


configs = [(2, 5), (4, 5), (8, 10), (12, 10)]
rows = []

for num_qubits, num_layers in configs:
    sim_perf = PureStateVectorSimulator(build_program(num_qubits, num_layers))
    perf_params = jnp.zeros(num_qubits * num_layers * 3)
    eager = best_of(sim_perf.compute, perf_params)
    jitted = best_of(jax.jit(sim_perf.compute), perf_params)
    rows.append((num_qubits, num_layers, eager, jitted))
    print(f"{num_qubits:2d} qubits x {num_layers:2d} layers:  "
          f"eager {eager * 1e3:9.2f} ms   jit {jitted * 1e3:9.2f} ms   "
          f"speedup {eager / jitted:6.1f}x")

In [ ]:
sizes = [r[0] for r in rows]
eager_ms = [r[2] * 1e3 for r in rows]
jit_ms = [r[3] * 1e3 for r in rows]

fig, (ax_time, ax_speed) = plt.subplots(1, 2, figsize=(11, 4))

ax_time.plot(sizes, eager_ms, "o-", label="eager")
ax_time.plot(sizes, jit_ms, "s-", label="jax.jit")
ax_time.set_yscale("log")
ax_time.set_xlabel("Qubits")
ax_time.set_ylabel("Time per call (ms)")
ax_time.set_title("State-vector evaluation cost")
ax_time.legend()
ax_time.grid(alpha=0.3)

ax_speed.plot(sizes, [e / j for e, j in zip(eager_ms, jit_ms)], "o-", color="tab:green")
ax_speed.axhline(1.0, color="grey", linestyle="--", linewidth=1)
ax_speed.set_yscale("log")
ax_speed.set_xlabel("Qubits")
ax_speed.set_ylabel(r"Speedup from $\mathtt{jax.jit}$")
ax_speed.set_title("JIT pays off most for small registers")
ax_speed.grid(alpha=0.3)

fig.tight_layout()
plt.show()


The trend is the point of the plot. For a handful of qubits, the runtime is almost entirely per-operation Python
overhead, which `jax.jit` eliminates. As the register grows, the $2^n$-amplitude state dominates the cost and the
advantage shrinks toward $1\times$. (This sweep stops at 12 qubits to keep the documentation build fast; extend
`configs` to see the falloff continue.)

The same reasoning applies to `DensityMatrixSimulator`. There the falloff sets in at roughly half the qubit count,
since a density matrix has $4^n$ entries rather than $2^n$.

In practice: always wrap `compute` in `jax.jit` when you evaluate a small-to-medium circuit repeatedly, as in the
variational loop of section 2. Don't expect it to rescue a circuit that is simply too large.